In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tifffile as tiff
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.layers import Add
import os
import math
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers,initializers,applications
import numpy as np
import sys
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
#为预测标签创建新路径
test_parameters = {
    "input_size": [3, 256, 256],                           #输入图片的shape
    "class_dim": -1,                                     #分类数
    "test_path":'E:/train_data/Semantic_segmentation/Gid_s/',       #原始数据集路径
    "target_path":'E:/train_data/Semantic_segmentation/Gid_s/src_5c_o/',        #要解压的路径 
    "test_list_path": "E:/train_data/Semantic_segmentation/Gid_s/tifs/gid_data.txt",              #train_data.txt路径
    "label_dict":{},                                    #标签字典
    "readme_path": "E:/train_data/Semantic_segmentation/Gid_s/tifs/readme.json",   #readme.json路径
    "num_epochs": 10,                                    #训练轮数
    "train_batch_size":64,                             #批次的大小
    "learning_strategy": {                              #优化函数相关的配置
        "lr": 0.01                                     #超参数学习率
    } 
}
def get_data_list(target_path,test_list_path):
    '''
    生成数据列表
    '''
    #存放所有类别的信息
    class_detail = []
    #获取所有类别保存的文件夹名称
    data_list_path=target_path
    class_dirs = os.listdir(data_list_path)
    if '__MACOSX' in class_dirs:
        class_dirs.remove('__MACOSX')
    # #总的图像数量
    all_class_images = 0
    # #存放类别数目
    class_dim = 0
    # #存储要写进test.txt中的内容
    test_list=[]
    #读取每个类别
    for class_dir in class_dirs:
        if class_dir != ".DS_Store":
            class_dim += 1
            #每个类别的信息
            class_detail_list = {}
            test_sum = 0
            #统计每个类别有多少张图片
      
            #获取类别路径 
            path = os.path.join(data_list_path,class_dir)
            # print(path)
            # 获取所有图片
            
            img_paths = os.listdir(path)
            for img_path in img_paths:  
                # 遍历文件夹下的每个图片
                if img_path =='.DS_Store':
                    continue
#                 name_path = os.path.join(path,img_path)            # 每张图片的路径
                name_path = img_path
                test_sum += 1 
                test_list.append(name_path + "\n") 
                print(name_path)
                #test_sum测试数据的数目
    
            
            # 说明的json文件的class_detail数据
            class_detail_list['class_name'] = class_dir             #类别名称
            class_detail_list['class_test_images'] = test_sum       #该类数据的测试集数目
            class_detail.append(class_detail_list)  
            
    #初始化分类数
    test_parameters['class_dim'] = class_dim
    #print(train_parameters)
#     random.shuffle(test_list)
#按照数字顺序排列
    test_list.sort(key=lambda x: int(x.split('.')[0]))
    with open(test_list_path, 'a') as f:
        for test_image in test_list:
            f.write(test_image) 

    # 说明的json文件信息
    readjson = {}
    readjson['all_class_name'] = data_list_path                  #文件父目录
    readjson['all_class_images'] = all_class_images
    readjson['class_detail'] = class_detail
    jsons = json.dumps(readjson, sort_keys=True, indent=4, separators=(',', ': '))
    with open(test_parameters['readme_path'],'w') as f:
        f.write(jsons)
    print ('生成数据列表完成！')

In [3]:
'''
参数初始化
'''
test_path=test_parameters['test_path']
target_path=test_parameters['target_path']
test_list_path=test_parameters['test_list_path']

#每次生成数据列表前，首先清空test.txt
with open(test_list_path, 'w') as f: 
    f.seek(0)
    f.truncate() 
#生成数据列表   
get_data_list(target_path,test_list_path)

1.tif
2.tif
3.tif
4.tif
5.tif
6.tif
7.tif
8.tif
生成数据列表完成！


In [4]:
COLORMAP = [
    [0, 0, 0],
    [255, 0, 0],
    [0, 255, 0],
    [0, 255, 255],
    [255, 255, 0],
    [0, 0, 255],
]

COLORCLASS = [
    'Background',
    'Building',
    'Farmland',
    'Forest',
    'Meadow',
    'Water',
]


COLORMAP1 = [
    [0, 0, 0],
    [200,0,0],
    [200,0,200],
    [250,0,150],
    [150,0,250],
    [200,150,150],
    [150,150,250],
    [250,150,150],
    [250,200,0],
    [0,200,0],
    [200,200,0],
    [150,250,0],
    [0,0,200],
    [150,200,150],
    [0,150,200],
    [0,200,250],
    
]
COLORCLASS1 = [
    'Background',
    'Industrial land',
    'Garden land',
    'Urban residential',
    'Arbor forest',
    'Rural residential',
    'Shrub land',
    'Traffic land',
    'Natural meadow',
    'Paddy field',
    'Artificial meadow',
    'Irrigated land',
    'River',
    'Dry cropland',
    'Lake',
    'Pond',
]



print('colormap size:', len(COLORMAP))
print('colorclass size:', len(COLORCLASS))

#第一张样本图像索引
colormap2label = np.zeros(256 ** 3)
for i, color_map in enumerate(COLORMAP):
#     print("dddd", (color_map[0] * 256 + color_map[1]) * 256 + color_map[2])
#     print(i)
    colormap2label[(color_map[0] * 256 + color_map[1]) * 256 + color_map[2]] = i
    #i = int(colormap2label[(color_map[0] * 256 + color_map[1]) * 256 + color_map[2]])
colormap2label = tf.convert_to_tensor(colormap2label)
print(colormap2label)
def label_indices(colormap,colormap2label):
    colormap = tf.cast(colormap, dtype=tf.int32)
    idx = ((colormap[:, :, 0] * 256 + colormap[:, :, 1]) * 256 + colormap[:, :, 2])
    #print(((colormap[:, :, 0] * 256 + colormap[:, :, 1]) * 256 + colormap[:, :, 2]).astype(np.int))
    return tf.gather_nd(colormap2label, tf.expand_dims(idx, -1))
#第一张样本图像索引
colormap2label1 = np.zeros(256 ** 3)
for i, color_map1 in enumerate(COLORMAP1):
#     print("dddd", (color_map[0] * 256 + color_map[1]) * 256 + color_map[2])
#     print(i)
    colormap2label1[(color_map1[0] * 256 + color_map1[1]) * 256 + color_map1[2]] = i
    #i = int(colormap2label[(color_map[0] * 256 + color_map[1]) * 256 + color_map[2]])
colormap2label1 = tf.convert_to_tensor(colormap2label1)
print(colormap2label)
def label_indices1(colormap1,colormap2label1):
    colormap1 = tf.cast(colormap1, dtype=tf.int32)
    idx = ((colormap1[:, :, 0] * 256 + colormap1[:, :, 1]) * 256 + colormap1[:, :, 2])
    #print(((colormap[:, :, 0] * 256 + colormap[:, :, 1]) * 256 + colormap[:, :, 2]).astype(np.int))
    return tf.gather_nd(colormap2label1, tf.expand_dims(idx, -1))

colormap size: 6
colorclass size: 6
tf.Tensor([0. 0. 0. ... 0. 0. 0.], shape=(16777216,), dtype=float64)
tf.Tensor([0. 0. 0. ... 0. 0. 0.], shape=(16777216,), dtype=float64)


In [5]:
isprs_dir = root=r'E:/train_data/Semantic_segmentation/Gid_s'

In [6]:
def read_images(is_train=True,root=isprs_dir):
    txt_fname = '%s//tifs//%s' % (
        root, 'gid_data.txt')
    with open(txt_fname, 'r') as f:
        image_names = f.read().split()
        
    label_images1 = [None] * len(image_names)
    label_images = [None] * len(image_names)

    for i, fname in enumerate(image_names):
        label_name = '%s\\label_5c\\%s' % (root, fname)
        label_name1 = '%s\\label_15c\\%s' % (root, fname)
        
        label_images[i] = tiff.imread(label_name)
        label_images1[i] = tiff.imread(label_name1)

    return label_images,label_images1

In [7]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from PIL import Image
import tifffile as tiff

# 读取图像
label_images, label_images1 = read_images(root)

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import tifffile as tiff
# 假设 read_images 和 label_indices 和 label_indices1 和 colormap2label 和 colormap2label1 已经定义

# 读取图像
# label_images, label_images1 = read_images(root)

# 初始化计数器
count_16 = np.zeros(16)
count_6 = np.zeros(6)

# 对每一张图像进行操作
# 假设您已经有了一个函数或方法来读取图像并获取它们的标签（如：read_images, label_indices, label_indices1）
for img_6, img_16 in zip(label_images, label_images1):
    labels_6 = label_indices(img_6, colormap2label).numpy()
    labels_16 = label_indices1(img_16, colormap2label1).numpy()

    # 更新计数器
    unique_16, counts_16 = np.unique(labels_16, return_counts=True)
    unique_6, counts_6 = np.unique(labels_6, return_counts=True)
    
    for u, c in zip(unique_16, counts_16):
        u = int(u)
        count_16[u] += c
    for u, c in zip(unique_6, counts_6):
        u = int(u)
        count_6[u] += c

# 保存各类别数量至Excel
df_6 = pd.DataFrame({'Class_6': COLORCLASS, 'Count_6': count_6})
df_16 = pd.DataFrame({'Class_16': COLORCLASS1, 'Count_16': count_16})
df_6.to_excel('G:/Fusion_model_excel/6_classes1.xlsx', index=False)
df_16.to_excel('G:/Fusion_model_excel/16_classes1.xlsx', index=False)


KeyboardInterrupt: 

In [ ]:
df = pd.read_excel("G:/Fusion_model_excel/matrix_6x16-1.xlsx", index_col=0)
matrix_6x16 = df.values

In [ ]:
row_sums = matrix_6x16.sum(axis=1, keepdims=True)
normalized_matrix_6x16 = matrix_6x16 / row_sums
print(normalized_matrix_6x16 )

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# 假设 read_images 已经定义
# label_images, label_images1 = read_images(root)

# 初始化6x16的矩阵
matrix_6x16 = np.zeros((6, 16))

# 对每一张图像进行操作
for img_6, img_16 in zip(label_images, label_images1):
    labels_6 = label_indices(img_6, colormap2label).numpy()
    labels_16 = label_indices1(img_16, colormap2label1).numpy()

    # 更新6x16的矩阵
    for i in range(6):
        for j in range(16):
            matrix_6x16[i, j] += np.sum((labels_6 == i) & (labels_16 == j))
            df = pd.DataFrame(matrix_6x16, index=COLORCLASS, columns=COLORCLASS1)

# 保存到Excel文件
df.to_excel("G:/Fusion_model_excel/matrix_6x16.xlsx")
